In [6]:
import os
import logging
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, Document
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_index.readers.web import SimpleWebPageReader

# --- Debug Logging ---
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- LLM and Embedding Initialization ---
llm = Ollama(model="gemma3:1b", temperature=0.0)
logger.info("Initialized local LLM via Ollama: gemma3:1b")

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
logger.info("Initialized HuggingFace embeddings")

# --- Apply Settings ---
Settings.llm = llm
Settings.embed_model = embed_model
logger.info("Configured global settings")

# --- Load & Parse Web Document ---
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
logger.info(f"Loading web page: {url}")

loader = SimpleWebPageReader()
docs = loader.load_data([url])
logger.info(f"Loaded {len(docs)} document(s)")

# --- Text Splitting ---
splitter = SentenceSplitter(chunk_size=1000, chunk_overlap=200)
nodes = splitter.get_nodes_from_documents(docs)
logger.info(f"Split into {len(nodes)} chunks")

# --- Indexing ---
index = VectorStoreIndex(nodes)
logger.info("Built vector index")

retriever = VectorIndexRetriever(index=index, similarity_top_k=4)
query_engine = RetrieverQueryEngine(retriever=retriever)

# --- Query ---
question = "What is Task Decomposition?"
logger.info(f"Querying: {question}")

response = query_engine.query(question)

# --- Output ---
print("\n=== Answer ===\n", response.response)
print("\n=== Source Nodes ===\n")
for node in response.source_nodes:
    print(node.node.text[:500], "\n---\n")

INFO:__main__:Initialized local LLM via Ollama: gemma3:1b
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']
INFO:__main__:Initialized HuggingFace embeddings
INFO:__main__:Configured global settings
INFO:__main__:Loading web page: https://lilianweng.github.io/posts/2023-06-23-agent/
INFO:__main__:Loaded 1 document(s)
INFO:__main__:Split into 44 chunks
INFO:__main__:Built vector index
INFO:__main__:Querying: What is Task Decomposition?
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



=== Answer ===
 The task decomposition process in agent systems is fundamentally about breaking down complex problems into smaller, manageable subgoals. It can be achieved through two primary methods: leveraging the LLM’s prompting capabilities – such as asking the model to outline the steps – or employing task-specific instructions, like providing detailed scripts or code. This decomposition strategy enhances efficiency and reduces complexity by tackling large problems into smaller, achievable units.


=== Source Nodes ===

It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.\nTask decomposition can be done (1) by LLM with simple prompting like \"Steps for XYZ.\\n1.\", \"What are the subgoals for achieving XYZ?\", (2) by using task-specific instructions; e